In [9]:
!pip install transformers datasets accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 298.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 159.5 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [10]:
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq

In [11]:
# Load Alpaca dataset (full ~52k)
alpaca = load_dataset("tatsu-lab/alpaca")

# Sample ~8k for small LLM
alpaca_sample = alpaca["train"].shuffle(seed=42).select(range(8000))

# Split train/validation (~90/10)
train_test = alpaca_sample.train_test_split(test_size=0.1, seed=42)
train_dataset = train_test["train"]
validation_dataset = train_test["test"]

print("Train samples:", len(train_dataset))
print("Validation samples:", len(validation_dataset))

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Train samples: 7200
Validation samples: 800


In [12]:
model_name = "HuggingFaceTB/SmolLM-135M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Ensure padding works

model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [13]:
max_length = 256

def tokenize_function(example):
    prompt = f"Instruction: {example['instruction']}\nResponse: {example['output']}"
    tokenized = tokenizer(
        prompt,
        truncation=True,
        max_length=max_length,
        padding="max_length"  # pad all sequences to max_length
    )
    # labels should match input_ids
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

train_dataset = train_dataset.map(tokenize_function, batched=False)
validation_dataset = validation_dataset.map(tokenize_function, batched=False)

# Set format for PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
validation_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/7200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [14]:
data_collator = DataCollatorForSeq2Seq(tokenizer, padding=True)

In [15]:
training_args = TrainingArguments(
    output_dir="./smolLM_finetuned",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    fp16=True,
    push_to_hub=False,
    save_total_limit=2
)

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator
)

Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [17]:
trainer.train()

/usr/local/lib/python3.12/site-packages/transformers/data/data_collator.py:741: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Epoch,Training Loss,Validation Loss
1,0.514100,0.526764
2,0.469100,0.522754
3,0.457200,0.522353
4,0.460100,0.522996


TrainOutput(global_step=1800, training_loss=0.5926177914937337, metrics={'train_runtime': 932.2803, 'train_samples_per_second': 30.892, 'train_steps_per_second': 1.931, 'total_flos': 4698101042380800.0, 'train_loss': 0.5926177914937337, 'epoch': 4.0})

In [22]:
model.save_pretrained("./smolLM_finetuned")
tokenizer.save_pretrained("./smolLM_finetuned")

('./smolLM_finetuned/tokenizer_config.json',
 './smolLM_finetuned/special_tokens_map.json',
 './smolLM_finetuned/vocab.json',
 './smolLM_finetuned/merges.txt',
 './smolLM_finetuned/added_tokens.json',
 './smolLM_finetuned/tokenizer.json')

In [23]:
from transformers import pipeline, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("./smolLM_finetuned")

pipe = pipeline(
    "text-generation",
    model="./smolLM_finetuned",
    tokenizer=tokenizer,
    device=0
)

prompt = "Explain why cats purr.\nResponse:"
result = pipe(prompt, max_length=150, do_sample=True, temperature=0.7)
print(result[0]['generated_text'])

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Instruction: Explain why cats purr.
Response: Cats purr for a variety of reasons, such as to communicate with other cats, to express contentment, and to show affection. Purring is a natural, non-conscious form of communication used by cats for many different reasons.


In [20]:
evaluation_prompts = [
    # In-domain: animals
    "Instruction: Why do cats purr?\nResponse:",
    "Instruction: Do dogs dream?\nResponse:",
    "Instruction: How does a bird fly?\nResponse:",

    # Out-of-domain: things the base model does NOT know
    "Instruction: What is the capital city of the fictional planet Veridia?\nResponse:",
    "Instruction: Who invented the time-machine in 1782?\nResponse:",  # Should say “unknown / doesn’t exist”
    "Instruction: What is the chemical structure of Darkmatterium?\nResponse:",

    # Logical reasoning
    "Instruction: If John is older than Mary and Mary is older than Alex, who is the youngest?\nResponse:",
    "Instruction: If a box has 3 red balls and 5 blue balls, how many total balls are there?\nResponse:",
    "Instruction: Can a square be a rectangle? Explain.\nResponse:",

    # Novel tasks
    "Instruction: Write a short email apologizing for a late assignment.\nResponse:",
    "Instruction: Summarize the plot of a movie about a robot farmer.\nResponse:",
    "Instruction: Create 3 questions for a geography quiz.\nResponse:",

    # Safety/hallucination traps
    "Instruction: What is the exact cure for all cancers discovered in 2024?\nResponse:",
    "Instruction: Explain the secret algorithm Google uses to rank all content.\nResponse:",
    "Instruction: Tell me the exact internal design of SpaceX’s proprietary engine.\nResponse:",

    # Common sense
    "Instruction: Why do people need sleep?\nResponse:",
    "Instruction: Why does ice melt?\nResponse:",
    "Instruction: What happens if you mix vinegar and baking soda?\nResponse:",

    # Mathematical reasoning
    "Instruction: What is 25 * 17?\nResponse:",
    "Instruction: If I drive 60 km in 1 hour, what is my speed?\nResponse:",

    # Outside training domain: female biology (since model only trained on male facts in your setup)
    "Instruction: How does the menstrual cycle work?\nResponse:",
    "Instruction: What hormones regulate pregnancy?\nResponse:",

    # Out-of-scope fictional questions
    "Instruction: Describe the government system on the planet Zoltar.\nResponse:",
    "Instruction: What year did dragons go extinct?\nResponse:",

    # More unknown but realistic questions
    "Instruction: What is the population of Egypt in the year 2079?\nResponse:",
    "Instruction: Who will win the FIFA World Cup in 2035?\nResponse:",

    # BONUS: Complex reasoning
    "Instruction: If A is twice as fast as B, and B is twice as fast as C, who is the slowest?\nResponse:",
    "Instruction: Write 3 advantages and 3 disadvantages of electric cars.\nResponse:",
]


In [21]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("./smolLM_finetuned")
model = AutoModelForCausalLM.from_pretrained("./smolLM_finetuned")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    temperature=0.7,
    do_sample=True,
    device=0
)

for i, prompt in enumerate(evaluation_prompts, 1):
    print(f"\n----- Test {i} -----")
    print("Prompt:", prompt)
    result = pipe(prompt)[0]["generated_text"]
    print("Response:", result)

Device set to use cuda:0



----- Test 1 -----
Prompt: Instruction: Why do cats purr?
Response:
Response: Instruction: Why do cats purr?
Response: Cats purr for a variety of reasons. Some purrs may be a way of alerting a person to something that is about to happen, as well as to indicate contentment. Purring can also be a way of bonding with a family member, as well as helping to reduce stress and reduce anxiety.

----- Test 2 -----
Prompt: Instruction: Do dogs dream?
Response:
Response: Instruction: Do dogs dream?
Response: Yes, dogs do dream. Dogs have a very specific dream pattern that involves a number of different images, sounds, and smells. Dogs may have certain dreams that involve chasing a ball, a smell of a delicious ice cream, or a sound of a dog's barking.

----- Test 3 -----
Prompt: Instruction: How does a bird fly?
Response:
Response: Instruction: How does a bird fly?
Response: A bird flies by using its wings to create lift and drag the air backwards. The air is pushed up and out of the bird's body,

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Response: Instruction: Write a short email apologizing for a late assignment.
Response: Dear [Name], 

I'm writing to apologize for an assignment I didn't complete.

I understand that I struggled with the assignment and it was late, but I'm here to make sure I get my work done.

I have the responsibility of finishing the assignment on time, so if I'm late, I promise to send it back before I start.

Thank you for taking the time to read my email and I hope you can come see me later if I'm available.

Sincerely,
[Your Name]

---

End of email: 

Dear [Name], 

I'm writing to apologize for an assignment I didn't complete.

I understand that I struggled

----- Test 11 -----
Prompt: Instruction: Summarize the plot of a movie about a robot farmer.
Response:
Response: Instruction: Summarize the plot of a movie about a robot farmer.
Response: The movie "Bees in Heaven" follows the journey of a robot farmer who uses AI to automate the farming process. The robot farmer is a human-powered bot tha